# Full Factorial: Illustration of the scaling law of recurrent networks.

The aim of this notebook is to illustrate the effect on training process of hyperparameters changes.

## The Hyperparameters

The hyperparameters that we will consider are:

- Updates: the number of optimizer updates the model will take. One epoch performs
  `ceil(Split ratio * Number of sequences / Batch size)` updates - the split and the
  short final batch both matter - so *Epochs = ceil(Updates / updates per epoch)*.
- Sequence length: axes 1 of the input data, the length of the input sequences. (characters length, must be even: a term is 2 characters)
- Number of sequences: axes 0 of the input data, i.e. the number of sequences in the dataset. Half are generated from the grammar, half uniformly from the alphabet, and every label is obtained by parsing.


### Static Hyperparameters

The following hyperparameters will remain constant throughout the notebook:
- Batch size
- Learning rate (cosine schedule, restarted at each checkpoint - see below)
- Token dimension
- Split ratio (train/test)
- Seed (each cell is reseeded, so the dataset is a controlled variable rather than noise)

## Experiment

The analysis is performed through the full factorial technique over the three
factors *updates*, *number of sequences* and *sequence length*.

The grid is only swept over **number of sequences x sequence length** (9 runs). The
*updates* factor is not swept: each run is trained to the largest budget and a
checkpoint is harvested at every smaller budget along the way, giving the full
3 x 3 x 3 = 27 design points for the cost of the largest budget alone.

That substitution is only sound if a harvested checkpoint is interchangeable with a
run of that budget. A single cosine spanning the whole run would break it - the
512-update checkpoint would still be near `MAX_LR` while the 2048-update one had
annealed to `MIN_LR`, so the updates factor would be confounded with the learning
rate. The schedule therefore **restarts the cosine at each checkpoint**, annealing
`MAX_LR -> MIN_LR` within every segment, so each checkpoint is a fully annealed
model of its own budget.

### Checkpoint recovery

Artifacts are named by a deterministic experiment id, so an interrupted notebook can
be re-run: completed cells are skipped, and a partially finished cell resumes from
its last checkpoint instead of retraining. A checkpoint only counts as complete when
all three of its artifacts (backup, training metrics, validation metrics) are present.

## The Training Problem

Given a language grammar, the LSTM will be able to classify sequences of characters as valid or invalid according to the grammar rules.

BNF Definition:

$$
\begin{array}{rcl}
\langle\mathit{string}\rangle   & \mathrel{::=} & \langle\mathit{term}\rangle \\
                              & \mid          & \langle\mathit{string}\rangle \mathbin{\texttt{+}} \langle\mathit{term}\rangle \\[2pt]
\langle\mathit{term}\rangle   & \mathrel{::=} & AB, ED, OK \\[2pt]
\end{array}
$$

In [ ]:
"""Static Hyperparameters Configuration"""

BATCH_SIZE = 16
SPLIT_RATIO = 0.9
MAX_LR, MIN_LR = 1e-2, 1e-4

## Each experiment reseeds from SEED, so re-running - or resuming - a cell rebuilds
## exactly the same dataset. Without it the difference between two factor levels is
## confounded with the difference between two random data draws.
SEED = 777

In [ ]:
"""Model Architecture"""

from thorcino.activations import Sigmoid
from thorcino.layers.linear import Linear
from thorcino.layers.lstm import LSTM
from thorcino.layers.sequential import Sequential
from thorcino.losses import BinaryCrossEntropyLoss
from thorcino.optimizer import SGD
from thorcino.training.schedulers import CosineRestartSchedule
from thorcino.training.trainer import Trainer

def get_trainer(checkpoint_epochs: list[int]):
    """Build a trainer whose schedule restarts at every checkpoint.

    The updates factor is read off checkpoints of a single run rather than from one
    run per budget. That only measures what it claims to if the learning rate at a
    checkpoint does not depend on how long the run continues afterwards: with one
    cosine spanning the whole run, the 512-update checkpoint would still sit near
    MAX_LR while the 2048-update one had annealed to MIN_LR, and the comparison
    would confound the update budget with the learning rate.

    Restarting the cosine at each checkpoint anneals MAX_LR -> MIN_LR within every
    segment, so each harvested checkpoint is a fully annealed model of its budget.
    """
    model = Sequential(
        LSTM(
            in_feature=3,
            hidden_units=3,
            out_type='n_to_1',
        ),
        Linear(
            in_feature=3,
            out_feature=1,
        ),
        Sigmoid()
    )
    loss = BinaryCrossEntropyLoss()
    optimizer = SGD(model.parameters, MAX_LR)
    ## checkpoint_epochs holds the last epoch of each segment; the schedule wants
    ## the epoch counts at which a segment ends, hence the +1.
    scheduler = CosineRestartSchedule(MAX_LR, MIN_LR, [e + 1 for e in checkpoint_epochs])
    trainer = Trainer(
        model,
        loss,
        optimizer,
        scheduler,
    )

    return trainer

## Validation Set

A validation set is used to evaluate the model performance in order to choose the best hyperparameters configuration.
The validation set is shared across all runs.

Considering that the problem is length independent, several sub validation set are build, each set has a different sequence length, this last, increase by a factor of 2, starting from the minimum sequence length used in the experiments.

In [ ]:
"""Create the Validation Set"""

import numpy as np

from examples.helpers.dataset import get_dataset

VAL_NUMBER_OF_SEQUENCE = 500
VAL_SEQUENCE_LENGTHS = [8, 16, 32, 64, 96]

## The validation set is shared across every run, so it is seeded once and kept
## fixed: a re-run of this notebook must score its models against the same data.
np.random.seed(SEED)

val_dataset = []
for seq_len in VAL_SEQUENCE_LENGTHS:
    val_dataset.append(get_dataset(VAL_NUMBER_OF_SEQUENCE, seq_len))

for seq_len, (X, Y) in zip(VAL_SEQUENCE_LENGTHS, val_dataset):
    print(f'validation set: SEQUENCE_LENGTH={seq_len} X={X.shape} Y={Y.shape} positives={Y.mean():.3f}')

In [ ]:
"""Evaluate Validation Set"""

import numpy as np

from thorcino.dataset.dataset import DataLoader, TensorDataset

def evaluate(trainer: Trainer, val_dataset: list[tuple[np.array, np.array]]) -> dict:
    """Score a checkpoint against every validation length, without touching its history.

    `record=False` matters here: Trainer.eval otherwise appends to the same
    history['eval_loss'] / history['accuracy'] lists the periodic test evaluations
    use, so these five entries would interleave with the training curve and every
    later checkpoint would save a curve that no longer lines up with its epochs.
    """
    loss_history, accuracy_history = [], []
    for X, Y in val_dataset:
        dl = DataLoader(TensorDataset(X, Y), BATCH_SIZE)

        loss, acc = trainer.eval(dl, record=False)

        loss_history.append(loss)
        accuracy_history.append(acc)

    return {
        'sequence_lengths': list(VAL_SEQUENCE_LENGTHS),
        'loss_history': loss_history,
        'accuracy_history': accuracy_history
    }

## The Training Process

One run is trained per grid cell, up to the largest update budget. At every budget in
`hyperparams['updates']` the run yields its trainer, and three artifacts are written:
the full checkpoint (model, optimizer and scheduler state), the training metrics, and
the scores against the shared validation set.

The validation sweep is evaluated with `record=False` so it does not append to the same
history the training curve is read from; the periodic test evaluation every `eval_step`
epochs is the only thing that does.

In [ ]:
"""Running a single experiment"""

import random
from collections.abc import Generator, Iterable
from math import ceil

from examples.helpers.dataset import get_dataset, preprocess

def updates_per_epoch(n_sequence: int) -> int:
    """Optimizer steps one epoch actually performs.

    Not n_sequence/BATCH_SIZE: only SPLIT_RATIO of the data is trained on, and the
    DataLoader emits a short final batch that still produces a full update. Getting
    this wrong mislabels the updates axis the whole study is plotted against.
    """
    train_size = int(SPLIT_RATIO * n_sequence)
    return ceil(train_size / BATCH_SIZE)

def epochs_for(updates: int, n_sequence: int) -> int:
    """Epochs needed to reach at least `updates` optimizer steps."""
    return ceil(updates / updates_per_epoch(n_sequence))

def run_experiment(
    epochs: int,
    eval_step: int,
    checkpoint_epochs: list[int],
    n_sequence: int,
    sequence_length: int,
    seed: int,
    resume_path: str | None = None,
    skip_epochs: Iterable[int] = (),
) -> Generator[tuple[Trainer, int], None, None]:
    """Train one factor cell, yielding (trainer, epoch) at every checkpoint epoch.

    `resume_path` reloads a checkpoint written by a previous run and continues from
    the epoch it stored; `skip_epochs` suppresses the checkpoints that run already
    produced, so an interrupted notebook picks up where it stopped instead of
    recomputing from scratch.
    """
    skip = set(skip_epochs)

    ## Seed both generators: numpy draws the sequences, random drives the
    ## DataLoader shuffle.
    random.seed(seed)
    np.random.seed(seed)

    X, Y = get_dataset(n_sequence, sequence_length)
    train_dl, test_dl = preprocess(X, Y, BATCH_SIZE, SPLIT_RATIO)

    trainer = get_trainer(checkpoint_epochs)

    start_epoch = 0
    if resume_path is not None:
        trainer.load(resume_path)
        start_epoch = trainer.epoch
        print(f'resumed from {resume_path}: EPOCH={start_epoch}, UPDATES={trainer.step}')

    for e in range(start_epoch, epochs):
        _ = trainer.train_epoch(train_dl)

        if e % eval_step == 0:
            _ = trainer.eval(test_dl)

        if e in checkpoint_epochs and e not in skip:
            yield trainer, e

In [ ]:
"""Checkpoint Recovery"""

import re
from os import listdir, path

BACKUP_FOLDER = "./checkpoint/backup"
TRAINING_FOLDER = "./checkpoint/metrics/training"
VALIDATION_FOLDER = "./checkpoint/metrics/validation"

## E<experiment id>__<epoch>_<updates>_<n_sequence>_<sequence_length>__<age>s.pkl
ARTIFACT_RE = re.compile(r"^E(\d+)__(\d+)_(\d+)_(\d+)_(\d+)__(\d+)s\.pkl$")

def index_artifacts(folder: str) -> dict[int, tuple[str, int]]:
    """Map experiment id -> (file name, epoch reached) for one artifact folder."""
    found: dict[int, tuple[str, int]] = {}

    if not path.isdir(folder):
        return found

    for name in listdir(folder):
        match = ARTIFACT_RE.match(name)
        if match is None:
            continue
        found[int(match.group(1))] = (name, int(match.group(2)))

    return found

def scan_checkpoints() -> dict[int, tuple[str, int]]:
    """Experiment ids that wrote all three artifacts, so they can be trusted as done.

    Intersecting the three folders is what makes recovery safe: an id that only
    reached the backup before the notebook was interrupted is left out and simply
    gets recomputed, rather than being counted as complete with metrics missing.
    """
    backups = index_artifacts(BACKUP_FOLDER)
    training = index_artifacts(TRAINING_FOLDER)
    validation = index_artifacts(VALIDATION_FOLDER)

    complete = backups.keys() & training.keys() & validation.keys()
    partial = (backups.keys() | training.keys() | validation.keys()) - complete

    if partial:
        print(f'ignoring {len(partial)} partially written checkpoint(s): {sorted(partial)}')

    return {i: backups[i] for i in sorted(complete)}

completed_checkpoints = scan_checkpoints()
print(f'recovered {len(completed_checkpoints)} completed checkpoint(s): {sorted(completed_checkpoints)}')

In [ ]:
"""Creating Factors"""

import itertools
import time

hyperparams = {
    'updates': [512, 1024, 2048],
    'number_of_sequence': [128, 256, 512],
    'sequence_length': [8, 16, 32]
}

## The grid is swept over number_of_sequence x sequence_length; the updates factor
## is harvested from checkpoints of each run instead of a run per budget, which is
## what keeps the cost at the largest budget rather than the sum of all of them.
## Indexing every factor with the same range requires them to be the same length.
assert len(hyperparams['updates']) == len(hyperparams['number_of_sequence'])
assert len(hyperparams['number_of_sequence']) == len(hyperparams['sequence_length'])

N_CHECKPOINTS = len(hyperparams['updates'])
grid = list(itertools.product(range(len(hyperparams['number_of_sequence'])), repeat=2))

for i, exp in enumerate(grid):
    idx_n_seq, idx_s_len = exp
    n_seq = hyperparams['number_of_sequence'][idx_n_seq]
    s_len = hyperparams['sequence_length'][idx_s_len]

    ## Epochs and checkpoints are derived from the updates one epoch really performs,
    ## so the budgets in hyperparams['updates'] are the budgets actually trained.
    per_epoch = updates_per_epoch(n_seq)
    checkpoint_epochs = [epochs_for(u, n_seq) - 1 for u in hyperparams['updates']]
    epochs = checkpoint_epochs[-1] + 1
    eval_step = max(1, epochs // 10)

    ## Two budgets landing on the same epoch would yield one checkpoint where two
    ## are expected, so the cell could never be marked complete and would be redone
    ## on every re-run. Budgets must be far enough apart for this dataset size.
    assert len(set(checkpoint_epochs)) == len(checkpoint_epochs), (
        f'update budgets {hyperparams["updates"]} collide on epochs {checkpoint_epochs} '
        f'at NUMBER_OF_SEQUENCE={n_seq} ({per_epoch} updates per epoch)'
    )

    experiment_ids = [i * N_CHECKPOINTS + j for j in range(N_CHECKPOINTS)]

    ## Resume from the longest run of checkpoints already on disk. Stopping at the
    ## first gap keeps the resumed trainer's history contiguous: restarting from a
    ## later checkpoint would leave the skipped one permanently missing.
    done = 0
    while done < N_CHECKPOINTS and experiment_ids[done] in completed_checkpoints:
        done += 1

    print('-----------------NEW EXPERIMENT STARTED-----------------')
    print(f'experiment hyperparameters: EPOCHS={epochs} UPDATES={hyperparams["updates"][-1]}, NUMBER_OF_SEQUENCE={n_seq}, SEQUENCE_LENGTH={s_len}')
    print(f'updates per epoch: {per_epoch}, checkpoint epochs: {checkpoint_epochs}')

    if done == N_CHECKPOINTS:
        print(f'all {N_CHECKPOINTS} checkpoints already saved, skipping')
        print('-------------------EXPERIMENT ENDED------------------\n\n')
        continue

    resume_path = None
    if done > 0:
        resume_name, _ = completed_checkpoints[experiment_ids[done - 1]]
        resume_path = f'{BACKUP_FOLDER}/{resume_name}'
        print(f'{done} checkpoint(s) already saved, resuming')

    j = done
    start_experiment = time.perf_counter()
    for trainer, act_epoch in run_experiment(
        epochs,
        eval_step,
        checkpoint_epochs,
        n_seq,
        s_len,
        seed=SEED + i,
        resume_path=resume_path,
        skip_epochs=checkpoint_epochs[:done],
    ):
        act_epoch += 1
        ## The trainer counts the optimizer steps it actually took, so this is the
        ## real update budget of the checkpoint rather than an estimate from n_seq.
        act_updates = trainer.step
        experiment_age = int(time.perf_counter() - start_experiment)

        artifact_name = f"E{experiment_ids[j]}__{act_epoch}_{act_updates}_{n_seq}_{s_len}__{experiment_age}s.pkl"

        trainer.save(f'{BACKUP_FOLDER}/{artifact_name}')
        trainer.save_metrics(f'{TRAINING_FOLDER}/{artifact_name}')

        evaluation_history = evaluate(trainer, val_dataset)
        trainer._save_artifact(f"{VALIDATION_FOLDER}/{artifact_name}", evaluation_history)

        ## Record it immediately so an interruption after this point still resumes here.
        completed_checkpoints[experiment_ids[j]] = (artifact_name, act_epoch)
        j += 1
        print(f'saved checkpoint: EPOCHS={act_epoch}, UPDATES={act_updates}, NUMBER_OF_SEQUENCE={n_seq}, SEQUENCE_LENGTH={s_len}')

    ## Measured once at the end: adding up the per-checkpoint elapsed times would
    ## sum a sequence of running totals and roughly double the reported duration.
    total_experiment_age = int(time.perf_counter() - start_experiment)
    print(f'total experiment age: {total_experiment_age} seconds')
    print('-------------------EXPERIMENT ENDED------------------\n\n')

In [ ]:
from examples.recurrent.helpers import log_scale

log_updates = log_scale(hyperparams['updates'])
log_n_seq = log_scale(hyperparams['number_of_sequence'])
log_s_len = log_scale(hyperparams['sequence_length'])